### Imports and Loading Model

In [1]:
# Import
from sentence_transformers import SentenceTransformer, util
import chromadb

In [2]:
# Load embedding model
model = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")

### Indexing Data

In [3]:
documents = [
    "The Eiffel Tower is in Paris.",
    "The Colosseum is in Rome.",
    "The Statue of Liberty is in New York.",
    "The Burj Khalifa is in Dubai."
]

# Setup Chroma client and collection
chroma_client = chromadb.Client()
# Drop existing collection if it already exists
try:
    chroma_client.delete_collection("demo")
except:
    pass

# Now create fresh
collection = chroma_client.create_collection("demo")

# Add documents with embeddings
embeddings = model.encode(documents).tolist()
collection.add(documents=documents, embeddings=embeddings, ids=[str(i) for i in range(len(documents))])

print("Indexed documents:", documents)

Indexed documents: ['The Eiffel Tower is in Paris.', 'The Colosseum is in Rome.', 'The Statue of Liberty is in New York.', 'The Burj Khalifa is in Dubai.']


### Basic RAG Query

In [4]:
query = "Where is the Eiffel Tower located?"
query_emb = model.encode([query]).tolist()

results = collection.query(query_embeddings=query_emb, n_results=2)
print("\nBasic RAG Results:", results["documents"][0])


Basic RAG Results: ['The Eiffel Tower is in Paris.', 'The Burj Khalifa is in Dubai.']


### Hybrid Search + Reranking

In [5]:
# Hybrid = keyword match + semantic search (simple demo: filter + semantic rerank)
keyword_matches = [doc for doc in documents if "Eiffel" in doc or "Paris" in doc]

# Re-rank keyword matches by semantic similarity
scores = [(doc, util.cos_sim(model.encode(query), model.encode(doc)).item()) for doc in keyword_matches]
reranked = sorted(scores, key=lambda x: x[1], reverse=True)

print("\nHybrid Search + Reranking Results:")
for doc, score in reranked:
    print(f"{doc} (score={score:.4f})")


Hybrid Search + Reranking Results:
The Eiffel Tower is in Paris. (score=0.8500)
